# StyleStudio — Adaptive Fusion smoke test (Kaggle GPU)

Notebook gọn, chỉ tập trung xác nhận các thay đổi trong `docs/superpowers/plans/2026-07-13-adaptive-end-fusion.md` (Task 0–6): dùng fork `namt9/StyleStudio`, nhánh `adaptive-fusion` — đã có `FusionController` (dừng fusion thích ứng thay vì `end_fusion` cố định) + fix bug `end_fusion` không forward vào `generate()`.

## ⚙️ TRƯỚC KHI CHẠY
1. **Settings → Accelerator → `GPU T4 x2`**. Nếu bị xám/khoá → xác thực số điện thoại tại `kaggle.com/settings` (mục Phone Verification), reload trang.
2. **Settings → Internet → `On`** (cũng cần xác thực SĐT).
3. (Khuyến nghị) **Add-ons → Secrets** → thêm secret `HF_TOKEN` = token HuggingFace (role Read, tạo tại `huggingface.co/settings/tokens`) → bật Attach. Tránh lỗi 403 khi tải model ẩn danh qua CDN Xet của HF.

Chạy lần lượt từng cell từ trên xuống.

## 1. Kiểm tra môi trường

In [ ]:
!nvidia-smi
import torch
print('torch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Chua bat GPU! Settings -> Accelerator -> GPU T4 x2 (xac thuc SDT neu bi xam)'

In [ ]:
import os
# Lay HF_TOKEN tu Kaggle Secrets neu co (Add-ons -> Secrets -> HF_TOKEN).
# Khong bat buoc, nhung giup tranh 403 khi tai model an danh qua CDN Xet cua HF.
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    print('Da lay HF_TOKEN tu Kaggle Secrets.')
except Exception as e:
    print('Khong co HF_TOKEN trong Secrets (bo qua neu chua can):', e)

## 2. Clone code (fork `namt9/StyleStudio`, nhánh `adaptive-fusion`)

In [ ]:
import os
REPO_DIR = '/kaggle/working/StyleStudio'
if not os.path.exists(REPO_DIR):
    !git clone -b adaptive-fusion https://github.com/namt9/StyleStudio.git {REPO_DIR}
%cd {REPO_DIR}
!git log --oneline -6
!ls

## 3. Cài dependencies
Giữ nguyên `torch` có sẵn của Kaggle (đã kèm CUDA), chỉ cài các thư viện được pin. KHÔNG cài lại torch/torchvision.

In [ ]:
!pip install -q \
  diffusers==0.25.1 \
  transformers==4.45.2 \
  tokenizers==0.20.1 \
  huggingface-hub==0.24.6 \
  accelerate peft safetensors einops omegaconf opencv-python pytest
print('done')

## 4. Tải checkpoint + vá đường dẫn cục bộ

- **SDXL base 1.0**: ưu tiên dùng **Kaggle Model** `stabilityai/stable-diffusion-xl` (Framework PyTorch, Variation `base-1-0`) — hạ tầng GCS riêng của Kaggle, không qua CDN Xet của HuggingFace đang lỗi. Cần **Add Input → Models** đính kèm model này trước (xem hướng dẫn). Nếu chưa attach, cell dưới tự fallback về tải từ HuggingFace (có retry, nhưng có thể vẫn 403 nếu sự cố Xet còn tiếp diễn).
- `madebyollin/sdxl-vae-fp16-fix`: repo nhỏ, tải bình thường qua HuggingFace.
- **Image encoder** (`h94/IP-Adapter`) và **CSGO ckpt** (`InstantX/CSGO`): source dùng như đường dẫn cục bộ → tải trước rồi trỏ lại.
- Xoá dòng HF mirror Trung Quốc (`hf-mirror.com`) — làm chậm/lỗi trên Kaggle.

> **Bối cảnh:** lỗi `403` trên `xet-bridge`/`cas-bridge.xethub.hf.co` là sự cố hạ tầng đang diễn ra của HuggingFace trên GCP (xảy ra cả khi đã đăng nhập token): [Xet-backed downloads stall on GCP](https://github.com/huggingface/xet-core/issues/800), [HF_HUB_DISABLE_XET không có tác dụng ở một số bản](https://github.com/huggingface/huggingface_hub/issues/3266), [Cas-bridge broke](https://discuss.huggingface.co/t/cas-bridge-xethub-hf-co-broke/158626). Dùng Kaggle Model tránh hẳn vấn đề này cho riêng file SDXL base (file lớn nhất, hay lỗi nhất).

In [ ]:
import os, time, glob

# Tat backend "Xet" cua huggingface_hub - hay bi 403 khi tai an danh (khong token)
# tren cac repo lon nhu SDXL base. Phai dat TRUOC moi lan import/goi huggingface_hub.
os.environ['HF_HUB_DISABLE_XET'] = '1'
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'
!pip uninstall -y hf_xet -q

from huggingface_hub import hf_hub_download, snapshot_download, login

HF_TOKEN = os.environ.get('HF_TOKEN', '')
if HF_TOKEN:
    login(token=HF_TOKEN)
    print('Da dang nhap HuggingFace bang token.')
else:
    print('CANH BAO: chua co HF_TOKEN. Neu gap loi 403 khi tai model, tao token (role Read) '
          'tai https://huggingface.co/settings/tokens roi them vao Kaggle Secrets (xem cell dau).')

# Bo qua anh preview/docs/openvino khong can thiet cho pipeline PyTorch
# (giam so file phai tai, giam rui ro trung blob dang loi tren CDN Xet)
SKIP_ASSETS = ['*.png', '*.jpg', '*.jpeg', '*.gif', '*.md', '*.PNG', '*.JPG',
               '*openvino*', 'onnx/*', '*.onnx']

def snapshot_download_retry(repo_id, retries=10, delay=20, **kw):
    """max_workers=1 (tuan tu) de tranh CDN rate-limit/403 khi tai song song.
    Retry dai + backoff tang dan vi loi 403 tren xet-bridge hien la su co ha tang
    HuggingFace tren GCP, chap chon theo edge server (xem link trong markdown tren)."""
    for attempt in range(1, retries + 1):
        try:
            return snapshot_download(repo_id, max_workers=1, **kw)
        except Exception as e:
            wait = delay * attempt
            print(f'  [{repo_id}] lan {attempt}/{retries} loi: {e}')
            if attempt == retries:
                raise
            print(f'  cho {wait}s roi thu lai...')
            time.sleep(wait)

# --- SDXL base: uu tien Kaggle Model (khong qua CDN Xet dang loi), fallback ve HuggingFace ---
BASE_MODEL_LOCAL = None
kaggle_model_hits = glob.glob('/kaggle/input/**/model_index.json', recursive=True)
if kaggle_model_hits:
    BASE_MODEL_LOCAL = os.path.dirname(kaggle_model_hits[0])
    print('Dung SDXL base tu Kaggle Model:', BASE_MODEL_LOCAL)
else:
    print('Khong thay Kaggle Model SDXL base da attach -> tai qua HuggingFace (co the cham/403).')
    print('Neu muon tranh 403: Add Input -> Models -> stabilityai/stable-diffusion-xl '
          '(Framework PyTorch, Variation base-1-0) roi chay lai cell nay.')
    snapshot_download_retry('stabilityai/stable-diffusion-xl-base-1.0', ignore_patterns=SKIP_ASSETS)
    print('Da cache SDXL base qua HuggingFace.')

snapshot_download_retry('madebyollin/sdxl-vae-fp16-fix', ignore_patterns=SKIP_ASSETS)
print('Da cache VAE.')

ie_root = snapshot_download_retry('h94/IP-Adapter', allow_patterns=['sdxl_models/image_encoder/*'])
IMAGE_ENCODER_LOCAL = os.path.join(ie_root, 'sdxl_models', 'image_encoder')
print('image_encoder:', IMAGE_ENCODER_LOCAL)

CSGO_LOCAL = hf_hub_download('InstantX/CSGO', 'csgo_4_32.bin')
print('csgo ckpt   :', CSGO_LOCAL)

def patch_paths(path):
    with open(path) as f:
        s = f.read()
    s = s.replace("os.environ['HF_ENDPOINT']='https://hf-mirror.com'",
                  "# HF mirror removed for Kaggle")
    if BASE_MODEL_LOCAL:
        s = s.replace('base_model_path = "stabilityai/stable-diffusion-xl-base-1.0"',
                      f'base_model_path = "{BASE_MODEL_LOCAL}"')
    s = s.replace('image_encoder_path = "h94/IP-Adapter/sdxl_models/image_encoder"',
                  f'image_encoder_path = "{IMAGE_ENCODER_LOCAL}"')
    s = s.replace("csgo_ckpt ='InstantX/CSGO/csgo_4_32.bin'",
                  f"csgo_ckpt ='{CSGO_LOCAL}'")
    with open(path, 'w') as f:
        f.write(s)
    print('patched:', path)

for p in ['infer_StyleStudio.py', 'infer_StyleStudio_layout_stability.py']:
    if os.path.exists(p):
        patch_paths(p)

In [ ]:
# --- Fix bug goc: infer_StyleStudio.py khong forward --end_fusion vao generate() ---
# generate() co default cung end_fusion=20; guard `if end_fusion != self.end_fusion:
# set_endFusion(...)` se GHI DE gia tri constructor bang 20. Hau qua: moi --end_fusion
# (tru dung 20) deu quy ve 20 -> truyen bao nhieu cung ra anh nhu nhau.
# Luu y: 'end_fusion=args.end_fusion' con xuat hien o loi goi constructor (co san o ban
# goc), nen phai kiem dung dong forward NGAY SAU num_inference_steps trong generate().
_p = 'infer_StyleStudio.py'
with open(_p) as f:
    _s = f.read()
_anchor = '        num_inference_steps=args.num_inference_steps,\n'
_fixed  = _anchor + '        end_fusion=args.end_fusion,\n'
if _fixed in _s:
    print('OK: infer da forward end_fusion -> generate() (khong can va).')
else:
    assert _anchor in _s, 'Khong thay anchor num_inference_steps trong generate() - kiem tra tay!'
    _s = _s.replace(_anchor, _fixed, 1)
    with open(_p, 'w') as f:
        f.write(_s)
    print('FIXED: da forward end_fusion -> generate() trong', _p)

## 5. Chạy unit test (Task 1–5: FusionController, attention processor, adapter/CLI, runner, eval)

In [ ]:
!python -m pytest tests/ -v

## 6. Smoke test: `--end_fusion` có tác dụng không? + Adaptive Fusion

25 bước cho nhanh. **Mục tiêu chính: kiểm chứng bug gốc đã fix** — truyền `--end_fusion` khác nhau phải ra ảnh khác nhau (trước fix mọi giá trị bị quy về 20).

Tiêu chí PASS:
1. Mục 5 (pytest) xanh hết.
2. **`mean|fixed5 − fixed20|` > ~1.0** (thang 0–255, cùng seed nên chỉ khác do `end_fusion`): chứng tỏ `--end_fusion` đã có tác dụng. Nếu ≈ 0 → vẫn dính bug.
3. Case adaptive không OOM; ghi nhận `stop_step` + `r(t)` từ log. *Lưu ý:* hiện `r(t)` chưa giảm (≈1.0) nên adaptive fuse tới trần `end_fusion_max`/số bước — đây là vấn đề **tín hiệu hội tụ**, tách riêng khỏi bug forward `end_fusion`.
4. `elapsed_sec` các case không chênh bất thường.

In [ ]:
import subprocess, os, json, shutil
import numpy as np
from PIL import Image

os.makedirs('smoke', exist_ok=True)

def run_case(name, extra_args):
    cmd = (f'HF_HUB_DISABLE_XET=1 PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python infer_StyleStudio.py '
           f'--prompt "A goat is playing on the beach" --style_path "assets/style1.jpg" '
           f'--adainIP --fuSAttn --num_inference_steps 25 '
           f'--log_json smoke/{name}.json {extra_args}')
    subprocess.run(cmd, shell=True, check=True)
    shutil.copy('test.jpg', f'smoke/{name}.jpg')
    print('done:', name)

# Kiem chung bug goc: cung seed, chi khac --end_fusion. Neu param co tac dung,
# fixed5 (fuse 5/25 buoc) va fixed20 (fuse 20/25) PHAI khac nhau ro ret.
run_case('fixed5',   '--end_fusion 5')
run_case('fixed20',  '--end_fusion 20')
run_case('adaptive', '--adaptive_fusion --rho 0.2 --end_fusion_max 30')

# So sanh dinh luong fixed5 vs fixed20
a = np.asarray(Image.open('smoke/fixed5.jpg').convert('RGB'), dtype=np.float32)
b = np.asarray(Image.open('smoke/fixed20.jpg').convert('RGB'), dtype=np.float32)
mad = float(np.abs(a - b).mean())
print(f"\n[verify] mean|fixed5 - fixed20| = {mad:.3f} (thang 0-255)")
print("  -> >~1.0: --end_fusion CO tac dung, bug goc da fix." if mad > 1.0
      else "  -> ~0: VAN DINH BUG, moi --end_fusion quy ve cung gia tri!")

for name in ['fixed5', 'fixed20', 'adaptive']:
    d = json.load(open(f'smoke/{name}.json'))
    fusion = d.get('fusion') or {}
    print(f"\n[{name}] elapsed={d['elapsed_sec']}s  stop_step={fusion.get('stop_step')}")
    if fusion.get('r_history'):
        print('  r(t):', [round(r, 3) for _, r in fusion['r_history']])

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

panels = [
    ('assets/style1.jpg', 'Style'),
    ('smoke/fixed5.jpg',  'end_fusion=5'),
    ('smoke/fixed20.jpg', 'end_fusion=20'),
    ('smoke/adaptive.jpg', 'adaptive_fusion'),
]
fig, ax = plt.subplots(1, len(panels), figsize=(5 * len(panels), 5))
for a, (path, title) in zip(ax, panels):
    a.imshow(Image.open(path)); a.set_title(title); a.axis('off')
plt.tight_layout(); plt.show()

## 🛠️ Xử lý lỗi thường gặp

- **GPU/Internet bị xám, không chọn được**: tài khoản Kaggle chưa xác thực số điện thoại — `kaggle.com/settings` → Phone Verification → xong reload trang.
- **`pip install` báo `from versions: none`**: không có Internet trong session (không phải sai version) — bật **Settings → Internet → On** rồi chạy lại.
- **`403 Forbidden` khi tải model, URL có `xet-bridge`**: lỗi CDN Xet của HuggingFace khi tải ẩn danh — thêm `HF_TOKEN` vào Kaggle Secrets (xem cell đầu mục 1) rồi **Restart session**, chạy lại từ mục 4. Mục 4 đã tự bỏ qua ảnh/docs không cần thiết và retry tuần tự.
- **`CUDA out of memory`**: giảm `--num_inference_steps`, hoặc bỏ `--fuSAttn` (batch=1). `enable_attention_slicing()` có thể bị `set_ip_adapter()` ghi đè (nó thay toàn bộ attention processor) — không phải mitigation chắc chắn.
- **Phiên Kaggle**: tối đa 12 giờ/phiên, quota 30 giờ GPU/tuần.